In [ ]:
# PDF 문서 불러오기

from langchain_community.document_loaders import PyPDFLoader

# PDF 파일 경로
file_path = "D:/shproject/ex0928/data/가상Tech_업무가이드.pdf"

# PDF 로더 생성 및 문서 불러오기
loader = PyPDFLoader(file_path)
documents = loader.load()

# 내용이 없는 페이지 제거
documents = [
    doc for doc in documents
    if doc.page_content.strip()
]

# 로드 결과 확인
print(f"전체 페이지 수: {len(documents)}")
print("\n첫 번째 페이지 내용:")
print(documents[0].page_content[:300])

In [ ]:
# PDF 문서 청킹

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서를 일정한 크기의 청크로 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    add_start_index=True,
)

docs = text_splitter.split_documents(documents)

# 청킹 결과 확인
print(f"전체 청크 수: {len(docs)}")
print(f"첫 번째 청크 길이: {len(docs[0].page_content)}")
print(f"첫 번째 청크 메타데이터: {docs[0].metadata}")

print("\n첫 번째 청크 내용:")
print(docs[0].page_content[:500])

In [ ]:
# Embedding 모델 지정, FAISS 벡터 DB 저장, Retriever 생성

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 임베딩 모델 생성
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# 청크를 임베딩하여 FAISS 벡터 DB에 저장
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

# 관련도가 높은 청크 3개를 가져오는 Retriever 생성
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
# LLM 객체 생성 및 Prompt 지정

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
다음 문서를 참고하여 질문에 답변하세요.
문서에 없는 내용은 추측하거나 임의로 만들지 말고, 모른다고 답변하세요.

문서:
{context}

질문:
{question}

답변:
""")

In [ ]:
# 체인 생성

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# TXT 파일의 질문을 하나씩 불러와 RAG 체인에 입력

import re

file_path = "D:/shproject/ex0928/data/주어진 질문 10개.txt"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# 번호가 붙은 질문만 추출
questions = []

for line in lines:
    line = line.strip()

    if re.match(r"^\d+\.", line):
        question = re.sub(r"^\d+\.\s*", "", line)
        questions.append(question)

# 질문을 하나씩 RAG 체인에 입력
for i, question in enumerate(questions, start=1):
    answer = chain.invoke(question)

    print(f"[질문 {i}]")
    print(question)

    print(f"\n[답변 {i}]")
    print(answer)

    print("\n" + "=" * 50 + "\n")